In [88]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



In [89]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()

,ID,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002174,1,144.0,41.0,0,0,0,1,3,3,...,0.517172,0.375126,3.325332,0.002314,3880771.500,473.464852,0.000768,0.182615,0.030508,0.000758
1,TRG002178,0,142.0,39.0,1,1,0,0,3,3,...,0.444391,0.444391,3.032144,0.005612,2372009.744,59.459710,0.004383,0.032012,0.001006,0.003685
2,TRG002204,1,135.0,31.0,0,0,0,1,2,1,...,0.534549,0.534549,2.485848,0.006752,1540027.421,33.935384,0.007584,0.024062,0.000529,0.006447
3,TRG002206,0,12.0,35.0,0,0,0,1,3,3,...,0.506185,0.506185,2.606255,0.003755,6936740.794,46.859265,0.005424,0.013707,0.000178,0.004543
4,TRG002210,0,109.0,61.0,1,0,0,0,2,1,...,0.462282,0.462282,2.809279,0.006521,1265399.054,39.621023,0.006585,0.034148,0.001083,0.005626


In [90]:
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 121 entries, ID to original_ngtdm_Strength
dtypes: float64(108), int64(12), object(1)
memory usage: 378.3+ KB


,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.00000,...,4.000000e+02,4.000000e+02,4.000000e+02,400.000000,4.000000e+02,400.000000,400.000000,400.000000,400.000000,400.000000
mean,12.697500,56.000208,51.804674,0.547500,2.902500,2.797500,2.830000,9.875000,6.562500,8.63250,...,3.957637e-01,3.911005e-01,2.722189e+00,0.003347,5.679717e+07,178.311246,32500.032620,0.056935,0.005965,0.029322
std,111.107417,27.137584,10.948522,0.498362,49.932114,49.937068,49.935558,86.092911,70.444284,86.20034,...,1.666319e-01,1.615922e-01,7.648849e-01,0.002419,7.063846e+08,1045.453432,177545.921568,0.047179,0.008379,0.115915
min,0.000000,0.000000,23.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.00000,...,7.050000e-11,7.050000e-11,-3.200000e-16,0.000008,0.000000e+00,0.000000,0.000248,0.000000,0.000000,0.000000
25%,0.000000,38.000000,44.516769,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.00000,...,3.199017e-01,3.184398e-01,2.340783e+00,0.001389,1.030473e+06,18.760570,0.001826,0.018628,0.000310,0.001464
50%,0.000000,55.000000,51.019507,1.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.00000,...,4.095627e-01,4.054695e-01,2.814884e+00,0.002944,3.277334e+06,67.929659,0.004383,0.047740,0.002330,0.003276
75%,0.000000,73.000000,60.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,1.00000,...,5.000049e-01,4.956920e-01,3.304411e+00,0.004798,9.079686e+06,157.370294,0.013769,0.085321,0.007962,0.009479
max,999.000000,144.000000,79.603012,1.000000,999.000000,999.000000,999.000000,999.000000,999.000000,999.00000,...,8.773779e-01,8.571429e-01,4.947427e+00,0.011301,1.390001e+10,20764.693790,1000000.000000,0.285100,0.060742,1.145601


In [91]:
df.replace(999, pd.NA, inplace=True)

In [92]:
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary before droppig "pCR" missing values rows and "RelapseFreeSurvival" column
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

summary

,Column,MissingCount,Dtype
0,pCR (outcome),5,object
1,PgR,1,object
2,HER2,1,object
3,TrippleNegative,1,object
4,ChemoGrade,3,object
5,Proliferation,2,object
6,HistologyType,3,object
7,LNStatus,1,object
8,Gene,88,object


In [93]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape

(395, 120)

In [94]:
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())


--- Missing Value Check ---
0


In [118]:

y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

In [119]:
keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)

rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean
)


In [120]:
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV
from scipy.stats import loguniform

# Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector), 
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_


Fitting 25 folds for each of 50 candidates, totalling 1250 fits
Best params: {'svm__C': 361.2478500429091, 'svm__class_weight': 'balanced', 'svm__gamma': 0.00037961668958008145}
Best CV ROC AUC: 0.6903497981933501


In [121]:
# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)
# y_pred_best = (scores_test >= thr_bal).astype(int)

# Metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))

Test ROC AUC: 0.7638297872340426
Test PR AUC: 0.5205332717241044
Test balanced accuracy: 0.7257446808510639
Confusion matrix (default threshold):
 [[65 29]
 [ 6 19]]
Classification report (default threshold):
               precision    recall  f1-score   support

           0       0.92      0.69      0.79        94
           1       0.40      0.76      0.52        25

    accuracy                           0.71       119
   macro avg       0.66      0.73      0.65       119
weighted avg       0.81      0.71      0.73       119

